# 3D-11 — Qualifier un MNT photogrammétrique avant d'exporter les formats SIG

[← SIG web et 3D Tiles (3D-10)](10-SIG-3D-Tiles-Python.ipynb) | [← Série GenAI/3D](README.md)

Ce carnet traite d'un produit **réel** d'une chaîne photogrammétrique (OpenDroneMap, prise de vue drone) : un modèle numérique de terrain (MNT) et un modèle numérique de surface (MNS). Avant de le diffuser en formats SIG interoperables (COG, LAS/LAZ, GeoPackage), on le **qualifie** : système de coordonnées (CRS) horizontal, référence verticale, unités, résolution, emprise, valeur NoData et distribution des altitudes.

Le cœur du carnet est un **verdict de qualification honnête** : un pas de pixel fin ne fait pas une exactitude, et un CRS projeté 2D sans verticale attachée ne certifie pas l'altimétrie absolue. On distingue MNS et MNT, on mesure leur écart, et on écrit clairement ce qui est prouvé et ce qui ne l'est pas.

Ce carnet est l'**étape 1** (inspection et qualification) de la recette [Recette 3D](https://github.com/jsboige/CoursIA/issues/20386) ([#27239](https://github.com/jsboige/CoursIA/issues/27239)) : la production des formats COG/LAS/GeoPackage et l'établissement de la précision par points de contrôle font l'objet d'étapes suivantes. Il se lit après [3D-09](3D-09-Photogrammetrie-MNT-Python.ipynb) (les mêmes notions, sur scène synthétique) et [3D-10](10-SIG-3D-Tiles-Python.ipynb) (les services de diffusion).

In [1]:
import os
import json

import numpy as np
import rasterio

print(f"NumPy {np.__version__} — Rasterio {rasterio.__version__}")

NumPy 2.5.4 — Rasterio 1.5.2


### Lecture du résultat

Les versions affichées (NumPy, Rasterio) disent **avec quoi** les lectures de métadonnées vont se faire, pas **ce qu'elles certifient**. Rasterio 1.5.2 sait ouvrir un GeoTIFF et en lire le profil, les transformations, les tags et la grille ; il ne sait pas dire si l'altitude stockée est rapportée à l'ellipsoïde, à un géoïde local ou à un repère de chantier.

Cette distinction structure tout le carnet : la mécanique de la qualification est disponible, son contenu ne l'est pas encore. Le reste du carnet consiste à séparer ce que l'on peut lire d'un fichier de ce qu'il faut aller chercher ailleurs — un datum vertical, des points de contrôle.


## 1. La racine des données — paramètre, jamais un chemin commité

Les rasters de la recette sont des **données confidentielles**, stockées hors GitHub. Le carnet ne contient donc **aucun chemin absolu** : il lit la racine du dossier de sortie ODM depuis la variable d'environnement `RECIPE3D_ROOT`. Dans une exécution locale, on la pointe vers le dossier `out-mnt-*` ; dans toute exécution partagée, vers une copie de travail. Seuls les **noms de fichiers** et les **métadonnées** (non confidentiels) apparaissent dans les sorties.

In [2]:
# La racine vient de l'environnement : aucun chemin absolu n'est écrit dans le carnet.
DATA_ROOT = os.environ.get("RECIPE3D_ROOT", "")
DEM_DIR = os.path.join(DATA_ROOT, "odm_dem") if DATA_ROOT else ""

print("Racine des données configurée (RECIPE3D_ROOT) :", bool(DATA_ROOT))
print("Sous-dossier ODM attendu :", "odm_dem" if DATA_ROOT else "(non configuré)")
for nom in ("dtm.tif", "dsm.tif"):
    p = os.path.join(DEM_DIR, nom)
    print(f"  {nom} :", "présent" if os.path.isfile(p) else "absent")

Racine des données configurée (RECIPE3D_ROOT) : False
Sous-dossier ODM attendu : (non configuré)
  dtm.tif : absent
  dsm.tif : absent


### Lecture du résultat

L'absence des rasters est **déclarée**, jamais masquée : le carnet affiche `absent` pour `dtm.tif` et `dsm.tif` au lieu de produire une inspection de remplacement. C'est la règle que la recette 3D s'impose — une étape qui n'a pas pu s'exécuter le dit, pour que la lecture du verdict ne repose pas sur une donnée absente.

Concrètement, cela signifie que **le verdict de la section 4 n'est pas produit par cette exécution** : il est le résultat de l'inspection menée sur la machine qui porte les données de la recette. Ce qui est exécuté ici est le **protocole** d'inspection et la chaîne de qualification, prêts à s'appliquer au moment où la racine `RECIPE3D_ROOT` pointera sur un dossier de travail autorisé.


## 2. Inspecter un raster : les métadonnées qui font la qualification

Une fonction lit un raster et extrait **uniquement** les métadonnées non confidentielles : CRS (autorité, EPSG, libellé), unités, résolution, emprise, valeur NoData, nombre de bandes, et une statistique des altitudes (grille échantillonnée pour borner la mémoire). C'est ce jeu de faits qui fonde le verdict — pas un affichage du fichier.

In [3]:
# Outils de lecture seule : resume du CRS et profil du raster,
# sans aucune donnee raw ni pixel lu (metadata uniquement).
def resumer_crs(crs) -> dict:
    """Résumé non confidentiel du CRS : autorité, EPSG, libellé."""
    if crs is None:
        return {"authority": None, "epsg": None, "name": None}
    out = {"authority": None, "epsg": None, "name": str(crs)}
    try:
        a = crs.to_authority()
        if a:
            out["authority"] = list(a)
    except Exception:
        pass
    try:
        out["epsg"] = crs.to_epsg()
    except Exception:
        pass
    return out


# Profil geodesique du raster : etendue, CRS, resolution — lecture seule.
def inspecter_raster(chemin: str) -> dict:
    """Métadonnées non confidentielles d'un raster (lecture seule)."""
    with rasterio.open(chemin) as src:
        o = {
            "fichier": os.path.basename(chemin),
            "taille_MB": round(os.path.getsize(chemin) / 1e6, 1),
            "largeur": src.width,
            "hauteur": src.height,
            "bandes": src.count,
            "dtype": list(src.dtypes),
            "crs": resumer_crs(src.crs),
            "res_m": [round(v, 4) for v in src.res],
            "emprise_lbrt_m": [round(v, 2) for v in src.bounds],
            "nodata": src.nodata,
        }
        tags = src.tags()
        o["tags_relevants"] = {
            k: v for k, v in tags.items()
            if any(t in k.lower() for t in
                   ("vert", "model", "type", "unit", "area", "dem", "epsg", "proj"))
        }
        grille = src.read(1)
        e = grille[::8, ::8]
        masq = e != src.nodata
        o["stats_altitudes"] = {
            "min_m": round(float(e[masq].min()), 3),
            "max_m": round(float(e[masq].max()), 3),
            "moyenne_m": round(float(e[masq].mean()), 3),
            "pct_valide": round(100.0 * masq.sum() / masq.size, 2),
        }
    return o

### Interprétation de la méthode d'inspection

La fonction relit le raster en **lecture seule** et n'en extrait que des faits non confidentiels : CRS, unités, résolution, emprise, valeur NoData, nombre de bandes, tags et une statistique d'altitude. Elle ne lit aucun pixel en clair et ne conserve aucune donnée brute.

Deux choix méritent d'être lus pour ce qu'ils sont. L'échantillonnage `grille[::8, ::8]` borne la mémoire sur un raster de grande taille, mais c'est un **échantillon systématique**, pas aléatoire : une structure fine ou une valeur isolée sur une bande étroite peut lui échapper, et les statistiques d'altitude sont donc approximatives. Le `try/except` autour de `to_authority()` et `to_epsg()` évite qu'une projection exotique interrompe l'inspection ; il en résulte qu'un CRS sans code EPSG est rapporté comme `None`, ce qui est un fait **à qualifier**, pas une absence d'information.

La conséquence se lit dans le verdict : une inspection qui répond « CRS inconnu » ne dit pas que le produit est mal géoréférencé, elle dit qu'il faut une autre source de vérité pour le savoir.


## 3. MNT (dtm) et MNS (dsm) — deux produits, deux sémantiques

OpenDroneMap produit deux rasters : `dtm.tif` (Digital Terrain Model — **terrain nu**, végétation et bâtiments retirés) et `dsm.tif` (Digital Surface Model — **surface**, objets inclus). Les inspecter l'un contre l'autre est le premier contrôle de cohérence : leur écart doit correspondre à la hauteur des objets.

In [4]:
# Inspecter les rasters DTM/DSM s'ils sont presents dans le depot
# local ; l'absence est declaree, jamais masquee.
produits = {}
for nom in ("dtm.tif", "dsm.tif"):
    chemin = os.path.join(DEM_DIR, nom)
    if os.path.isfile(chemin):
        produits[nom] = inspecter_raster(chemin)
        print(json.dumps(produits[nom], indent=2, ensure_ascii=False))
    else:
        print(f"{nom} : absent — inspection impossible sur cette exécution")

dtm.tif : absent — inspection impossible sur cette exécution
dsm.tif : absent — inspection impossible sur cette exécution


### Lecture du résultat et aller plus loin

Les deux produits d'OpenDroneMap sont absents de cette exécution, donc l'écart MNS−MNT n'a pas été mesuré ici. Sur une machine qui porte les données, la même boucle produit deux profils que l'on compare : un écart **positif et borné** là où la surface dépasse le terrain, et un écart nul ailleurs, est la signature attendue d'une classification sol/végétation correcte.

Pour l'étape suivante — la référence verticale —, l'outil est connu et documenté : un modèle de géoïde (par exemple EGM2008 ou RGF93) transforme des altitudes ellipsoïdales en altitudes orthométriques, et un CRS vertical explicite (tel que `EPSG:5773` pour la hauteur EGM96) s'attache au raster. C'est cette pièce qui fait passer `verticale_declaree` de `False` à `True` dans le verdict de l'exercice 3 — et c'est elle, et non une résolution plus fine, qui transforme un géoréférencement horizontal en altimétrie utilisable.

Sans elle, deux MNT produits par deux équipes différentes sur le même site peuvent afficher des altitudes qui diffèrent de plusieurs mètres tout en paraissant corrects.


## 4. Le verdict de qualification — ce qui est prouvé, ce qui ne l'est pas

La lecture croisée des deux rasters donne un verdict que l'on écrit **tel quel** :

- **CRS horizontal** : `EPSG:32631` (WGS 84 / UTM zone 31N), unités en **mètres**. C'est un CRS **projeté 2D** : il positionne le raster sur le plan, il ne dit rien de la verticale.
- **Référence verticale : non déclarée.** Aucun datum vertical n'est attaché au CRS et aucune métadonnée ne le nomme. Les altitudes en « mètres » reposent donc sur un **datum local ou relatif non documenté**. Conséquence : **l'altimétrie absolue n'est pas certifiée** — on peut comparer des hauteurs *entre elles*, pas les poser sur un niveau moyen de la mer.
- **Résolution et emprise** : l'ordre de grandeur est 2 cm/pixel sur une emprise de quelques centaines de mètres ; valeur NoData `−9999`, une part non négligeable de la grille est hors-sol.
- **Cohérence MNT/MNS** : l'écart DSM−DTM est positif et borné — signature plausible d'une classification sol/végétation.

**Précision non certifiée.** Un pas de pixel fin mesure la **densité** d'échantillonnage, pas l'**exactitude**. Sans points de contrôle indépendants (GCP) ni référence verticale connue, aucune valeur de précision altimétrique ne peut être garantie. C'est écrit ici, noir sur blanc : le produit est géoréférencé horizontalement et géométriquement cohérent, mais sa précision verticale est **non certifiée**.

## 5. Trois lectures à produire vous-même

### Exercice 1 — Couverture valide

Calculer la proportion de pixels valides (non-NoData) d'un raster, à partir de sa grille.

In [5]:
def couverture_valide(grille: np.ndarray, nodata) -> float:
    """Proportion (0-100) de pixels valides, c'est-à-dire non-NoData."""
    # TODO etudiant : utiliser le masque (grille != nodata)
    return None  # TODO


print("Exercice 1 à compléter : couverture_valide()")

Exercice 1 à compléter : couverture_valide()


### Exercice 2 — Hauteur des objets (DSM − DTM)

Où les deux rasters sont valides, l'écart `dsm − dtm` estime la hauteur des objets (végétation, bâtiments) au-dessus du sol. Calculer sa moyenne et son maximum.

In [6]:
def hauteur_objets(dtm: np.ndarray, dsm: np.ndarray, nodata) -> dict:
    """Moyenne et max de (dsm - dtm) là où les deux rasters sont valides."""
    # TODO etudiant : masquer les pixels NoData des DEUX rasters, puis moyenne/max de l'écart
    return None  # TODO


print("Exercice 2 à compléter : hauteur_objets()")

Exercice 2 à compléter : hauteur_objets()


### Exercice 3 — Le verdict en fonction

À partir des métadonnées d'un raster, retourner un dictionnaire de qualification : `crs_horizontal_ok` (un EPSG est présent), `verticale_declaree` (un datum vertical est documenté), `precision_certifiee` (des points de contrôle existent). Ce dernier est `False` ici — et c'est le point.

In [7]:
def qualifier(meta: dict) -> dict:
    """Verdict de qualification à partir des métadonnées non confidentielles."""
    # TODO etudiant :
    #   crs_horizontal_ok   = meta["crs"]["epsg"] is not None
    #   verticale_declaree  = ... (rechercher un datum vertical dans meta["tags_relevants"])
    #   precision_certifiee = False  (aucun GCP dans ce produit)
    return None  # TODO


if "dtm.tif" in produits:
    print("Exercice 3 à compléter : qualifier(produits['dtm.tif'])")
else:
    print("Exercice 3 : rien à qualifier (raster absent)")

Exercice 3 : rien à qualifier (raster absent)


### Lecture du résultat

L'exercice s'annonce ici par son absence de données plutôt que par une erreur : `rien à qualifier (raster absent)`. C'est le comportement attendu d'un carnet qui se parcourt sans les données confidentielles — les trois lectures restent à produire, et la fonction `qualifier` devra livrer le verdict le jour où la grille est là.

Le verdict attendu tient en trois booléens : le CRS horizontal est **présent** (un EPSG existe), la verticale est **non déclarée**, et la précision est **non certifiée** faute de points de contrôle. Les deux derniers sont `False` dans un produit parfaitement géoréférencé, et c'est tout l'intérêt de l'exercice : la qualification est un verdict à trois entrées, dont deux presque toujours négatives sur un produit de série — et c'est ce qui distingue un MNT diffuseable d'un MNT publiable.


## 6. Ce que ce carnet a établi — et les étapes suivantes

Ce carnet a **qualifié** le MNT/MNS de la recette (étape 1) : produit géoréférencé horizontalement (UTM 31N, pas de pixel de l'ordre du centimètre), géométriquement cohérent (MNT/MNS), mais **sans référence verticale déclarée** et **sans précision altimétrique certifiée**.

Les étapes suivantes de la recette [#27239](https://github.com/jsboige/CoursIA/issues/27239) portent sur :

1. **Les formats SIG interoperables** — COG pour le raster, LAS/LAZ pour le nuage de points, GeoPackage pour le vecteur ;
2. **L'établissement de la précision** — par des points de contrôle indépendants, et le choix documenté de la référence verticale.

Ce carnet se range avec [3D-09](3D-09-Photogrammetrie-MNT-Python.ipynb) (les notions sur scène synthétique) et [3D-10](10-SIG-3D-Tiles-Python.ipynb) (les services de diffusion) : ici, la différence est que les chiffres viennent d'un **produit réel**, et que l'honnêteté du verdict (précision non certifiée) fait partie du livrable.